# Part 4 — Race: NeSy vs no-NeSy (the capstone)

Put the two agents in the **same scene** and let them race — each overtakes the background traffic and tries to surpass the other — scoring both *who finishes first* **and** *who stays safe and rule-compliant under competitive pressure*.

- **Agent A — no-NeSy:** the frozen Part-1 baseline (`part1_best`).
- **Agent B — NeSy:** the Part-2 shielded + fine-tuned agent (`part2_nesy`).

Same start, same traffic, same seeds; each runs its own policy on its own local observation. Saves the race scorecard + **`part4_race.mp4`** to Drive.

> **Honest caveat:** a 'be ahead' incentive rewards aggression, so the scorecard always pairs finishing progress with crash + violation metrics — never the winner alone.

## 1. Setup + load both agents from Drive

In [ ]:
import os, warnings, logging
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=r'.*Gym has been unmaintained.*')
logging.getLogger('gym').setLevel(logging.ERROR)

from google.colab import drive
drive.mount('/content/drive')

TOKEN = None
try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = None
if not TOKEN:
    import getpass
    TOKEN = getpass.getpass('GitHub token (input hidden): ')
assert TOKEN, 'No token provided.'
os.environ['GITHUB_TOKEN'] = TOKEN
print('Token loaded (not displayed).')

In [ ]:
%%bash
set -e
GH_USER=silvergjeka22
REPO=nesy-highway-driving
curl -fsSL -H "Authorization: token ${GITHUB_TOKEN}" \
  "https://raw.githubusercontent.com/${GH_USER}/${REPO}/main/bash/setup_colab.sh" \
  -o /content/setup_colab.sh
bash /content/setup_colab.sh

In [ ]:
import sys, json
sys.path.insert(0, '/content/nesy-highway-driving')
%cd /content/nesy-highway-driving
from utils import load_config, drive_path, save_json
cfg = load_config('configs/highway.yaml')
from pyvirtualdisplay import Display
_display = Display(visible=0, size=(900, 480)); _display.start()

from agents.baselines import load_model
TAG = json.load(open(drive_path(cfg, 'metrics', 'part1_best.json')))['tag']
agent_a = load_model(drive_path(cfg, 'checkpoints', f'part1_best_{TAG}.zip'), TAG)   # no-NeSy
agent_b = load_model(drive_path(cfg, 'checkpoints', 'part2_nesy.zip'), TAG)          # NeSy
print('loaded A (baseline) and B (NeSy); algo =', TAG)

## 2. Run N races (same-scene multi-agent) and score per agent
`make_race_env` builds a multi-agent highway-env (`controlled_vehicles=2`). Agent B runs the FSM safety shield. Winner = `cfg['race']['winner_metric']` (default longitudinal `distance`). The scorecard reports progress **and** crashes + shield interventions.

In [ ]:
from eval.race import race
import pandas as pd
results = race(agent_a, agent_b, cfg)              # A = baseline, B = NeSy (shielded)
save_json(results, drive_path(cfg, 'metrics', 'part4_race.json'))
print('wins (A=baseline, B=NeSy):', results['wins'], '| metric:', results['metric'])
m = results['metric']
pd.DataFrame([{'seed': r['seed'],
               f'A_{m}': round(r['A'][m], 1), 'A_crash': r['A']['crashed'],
               f'B_{m}': round(r['B'][m], 1), 'B_crash': r['B']['crashed'],
               'B_intervene': round(r['B']['intervention_rate'], 3),
               'winner': r['winner']}
              for r in results['races']])

## 3. Save the race video to Drive

In [ ]:
from eval.race import record_race_video
video = record_race_video(agent_a, agent_b, cfg, drive_path(cfg, 'videos', 'part4_race.mp4'))
print('Race video (A=baseline, B=NeSy, one shared scene) saved to:', video)

**Capstone exit criterion.** In head-to-head racing, the NeSy agent finishes **as fast or faster** than the no-NeSy baseline while recording **fewer crashes and provably fewer rule violations** — with **`part4_race.mp4`** on Drive as the visual proof.

To race against the **MetaDrive** agent instead, set `agent_b = load_model(drive_path(cfg,'checkpoints','part3_metadrive.zip'), 'ppo')` and use the MetaDrive race env (Tier-2; see `eval/race.py` TODO).